# Shear Viewer Demonstration

This notebook demonstrates the new shear visualization wrappers exposed on `ShearCheck`:

1. `plot_cot_theta_study`
2. `plot_cot_theta_moment_shift_study`
3. `plot_link_angle_study`
4. `plot_link_angle_moment_shift_study`
5. `plot_cot_theta_link_angle_heatmap`
6. `plot_axial_cot_theta_contour`

All methods return Plotly figures.


In [1]:
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)
from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar, ShearRebar
from materials.reinforced_concrete.code_checks.ec2_2004 import ShearCheck, ShearLoadCase

## 1. Build a Section and Shear Check

In [2]:
# Geometry
width = 300.0
height = 500.0
cover = 35.0
link_dia = 10.0

section = create_rectangular_section(width=width, height=height)

# Longitudinal reinforcement
bot_bar = Rebar(diameter=20, grade="B500B")
top_bar = Rebar(diameter=16, grade="B500B")
side_cover = cover + link_dia
y_bot = cover + link_dia + bot_bar.diameter / 2.0
y_top = height - cover - link_dia - top_bar.diameter / 2.0

section.add_rebar_group(
    create_linear_rebar_layer(
        rebar=bot_bar,
        n_bars=4,
        start_point=(side_cover + bot_bar.diameter / 2.0, y_bot),
        end_point=(width - side_cover - bot_bar.diameter / 2.0, y_bot),
    )
)
section.add_rebar_group(
    create_linear_rebar_layer(
        rebar=top_bar,
        n_bars=2,
        start_point=(side_cover + top_bar.diameter / 2.0, y_top),
        end_point=(width - side_cover - top_bar.diameter / 2.0, y_top),
    )
)

# Materials and shear reinforcement
concrete = ConcreteMaterial(grade="C30/37")
links = ShearRebar(
    diameter=10,
    link_spacing=150,
    n_legs=2,
    angle=90.0,
    grade="B500B",
)

check = ShearCheck(
    section=section,
    concrete=concrete,
    shear_reinforcement=links,
    use_mechanical_lever_arm=True,
)

## 2. Define a Baseline Load Case

In [3]:
load_case = ShearLoadCase(V_Ed=220.0, M_Ed=80.0, N_Ed=150.0)
result = check.perform_check(load_case=load_case)

print(f"Utilization: {result.utilization:.3f}")
print(f"Governing component: {result.details['governing_component']}")
print(f"cot(theta): {result.details['cot_theta']}")

Utilization: 0.513
Governing component: V_Rd_s
cot(theta): 2.5


## 3. Cot(theta) Capacity Study

Shows `V_Rd,s` and `V_Rd,max` versus cot(theta), including design reference lines (`V_Rd,max` at cot(theta) min and `V_Rd,s` at cot(theta) max) and the crossover cot(theta).


In [4]:
fig_cot = check.plot_cot_theta_study(
    load_case=load_case,
    n_points=60,
    show=False,
)
fig_cot

## 4. Cot(theta) Moment-Shift Study

Shows utilization and tension-shift add-on moment (`M_add`) versus cot(theta).


In [5]:
fig_cot_shift = check.plot_cot_theta_moment_shift_study(
    load_case=load_case,
    n_points=60,
    show=False,
)
fig_cot_shift


## 5. Link Angle Study

Compares shear capacities while varying the stirrup angle for a fixed cot(theta).

In [6]:
fig_alpha = check.plot_link_angle_study(
    load_case=load_case,
    cot_theta=2.5,
    angle_min=45.0,
    angle_max=90.0,
    n_points=46,
    show=False,
)
fig_alpha

## 6. Link Angle Moment-Shift Study

Shows utilization and tension-shift add-on moment (`M_add`) versus link angle for a fixed cot(theta).

In [7]:
fig_alpha_shift = check.plot_link_angle_moment_shift_study(
    load_case=load_case,
    cot_theta=1.8,
    angle_min=45.0,
    angle_max=90.0,
    n_points=46,
    show=False,
)
fig_alpha_shift

## 7. Cot(theta) vs Link Angle Heatmap

Visual overview of utilization over the cot(theta)-alpha domain.

In [8]:
fig_heatmap = check.plot_cot_theta_link_angle_heatmap(
    load_case=load_case,
    metric="utilization",
    n_cot=40,
    n_angles=40,
    show=False,
)
fig_heatmap

In [9]:
fig_heatmap = check.plot_cot_theta_link_angle_heatmap(
    load_case=load_case,
    metric="capacity",
    n_cot=40,
    n_angles=40,
    show=False,
)
fig_heatmap

In [10]:
fig_heatmap = check.plot_cot_theta_link_angle_heatmap(
    load_case=load_case,
    metric="v_rd_s",
    n_cot=40,
    n_angles=40,
    show=False,
)
fig_heatmap

In [11]:
fig_heatmap = check.plot_cot_theta_link_angle_heatmap(
    load_case=load_case,
    metric="v_rd_max",
    n_cot=40,
    n_angles=40,
    show=False,
)
fig_heatmap

## 8. Axial Force vs Cot(theta) Contour

Useful for assessing sensitivity of utilization (or capacities) to changing axial force.

In [12]:
fig_axial = check.plot_axial_cot_theta_contour(
    load_case=load_case,
    N_min=-1000.0,
    N_max=2000.0,
    n_axial=35,
    n_cot=45,
    metric="utilization",
    show=False,
)
fig_axial

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\analysis\shear_viewer.py:153: UserWarning:

Effective depth fallback (no compression/tension split): d = 450.0 mm

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 341.6 mm (0.76d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 365.3 mm (0.81d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 384.8 mm (0.86d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:

In [13]:
fig_axial = check.plot_axial_cot_theta_contour(
    load_case=load_case,
    N_min=-400.0,
    N_max=1200.0,
    n_axial=35,
    n_cot=45,
    metric="capacity",
    show=False,
)
fig_axial

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm clamped to upper bound: z_mech=426.6 mm > 0.95d=422.8 mm.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm clamped to upper bound: z_mech=426.4 mm > 0.95d=422.8 mm.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 292.1 mm (0.66d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 286.9 mm (0.64d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials

In [14]:
fig_axial = check.plot_axial_cot_theta_contour(
    load_case=load_case,
    N_min=-400.0,
    N_max=1200.0,
    n_axial=35,
    n_cot=45,
    metric="v_rd_s",
    show=False,
)
fig_axial

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm clamped to upper bound: z_mech=426.6 mm > 0.95d=422.8 mm.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm clamped to upper bound: z_mech=426.4 mm > 0.95d=422.8 mm.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 292.1 mm (0.66d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 286.9 mm (0.64d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials

In [15]:
fig_axial = check.plot_axial_cot_theta_contour(
    load_case=load_case,
    N_min=-400.0,
    N_max=1200.0,
    n_axial=35,
    n_cot=45,
    metric="v_rd_max",
    show=False,
)
fig_axial

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm clamped to upper bound: z_mech=426.6 mm > 0.95d=422.8 mm.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm clamped to upper bound: z_mech=426.4 mm > 0.95d=422.8 mm.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 292.1 mm (0.66d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:647: UserWarning:

Lever arm fallback to 286.9 mm (0.64d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials

## 9. Optional: Save Plots to HTML

In [16]:
# _ = check.plot_cot_theta_study(
#     load_case=load_case,
#     show=False,
#     save_path="shear_cot_theta_study.html",
# )
# _ = check.plot_cot_theta_moment_shift_study(
#     load_case=load_case,
#     show=False,
#     save_path="shear_cot_theta_moment_shift_study.html",
# )
# _ = check.plot_link_angle_study(
#     load_case=load_case,
#     show=False,
#     save_path="shear_link_angle_study.html",
# )
# _ = check.plot_link_angle_moment_shift_study(
#     load_case=load_case,
#     show=False,
#     save_path="shear_link_angle_moment_shift_study.html",
# )
# print("Saved shear_cot_theta_study.html")
# print("Saved shear_cot_theta_moment_shift_study.html")
# print("Saved shear_link_angle_study.html")
# print("Saved shear_link_angle_moment_shift_study.html")
